多智能体场景中，需要1个身份标记，其他场景中，例如1流式输出归因、调试与trace可读性、组件化封装，前端真是与运行态可展示性

In [1]:
from typing import Literal

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os

from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool


load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)


#2 声明1个工具
class weatherInput(BaseModel):
    """
    查询天气工具的形参的类的相关信息
    """
    city: str=Field(description="城市名称，例如'北京市','重庆市'",default="重庆市")
    unit:Literal["celsius","kelvin"]=Field(description="描述气温的单位")
    date:str=Field(description="查询日期")

@tool(description="查询具体城市的天气",args_schema=weatherInput)
def get_weather(city:str,unit:Literal["celsius","kelvin"],date:str):
    return f"{city} {date}的天气晴朗，气温10-16{unit}，有微风，适宜出行"
#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    tools=[get_weather],
    name="agent01",
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错

        "messages":
            [
                SystemMessage("你是1个私人ai助手"),
                HumanMessage("请你查询一下重庆2026-9-15的天气")
        ]
    }
)
rprint(response)
# print(response)

{
    'messages': [
        SystemMessage(
            content='你是1个私人ai助手',
            additional_kwargs={},
            response_metadata={},
            id='07c44939-c344-48b0-9528-e2f38f66aee3'
        ),
        HumanMessage(
            content='请你查询一下重庆2026-9-15的天气',
            additional_kwargs={},
            response_metadata={},
            id='71b84eae-a0aa-4ad8-8f66-c1ed37904c05'
        ),
        AIMessage(
            content='我来帮您查询重庆2026年9月15日的天气情况。\n\n',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 184,
                    'prompt_tokens': 391,
                    'total_tokens': 575,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 106,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 184
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 391
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-d5fad03b-3a30-97d4-8ee6-faf85298452e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0a919-533a-7230-a767-d4f5be3d7449-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '重庆市', 'unit': 'celsius', 'date': '2026-9-15'},
                    'id': 'call_bb0ef3f30e954856b76e3bdf',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 391,
                'output_tokens': 184,
                'total_tokens': 575,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 106}
            }
        ),
        ToolMessage(
            content='重庆市 2026-9-15的天气晴朗，气温10-16celsius，有微风，适宜出行',
            name='get_weather',
            id='585cb600-c945-44bb-8ba1-f6649ff1fcab',
            tool_call_id='call_bb0ef3f30e954856b76e3bdf'
        ),
        AIMessage(
            content='查询结果如下：\n\n## 🌤️ 重庆市 2026年9月15日 天气\n\n| 项目 | 详情 |\n|------|------|\n| 
**天气状况** | ☀️ 晴朗 |\n| **气温** | 10℃ ~ 16℃ |\n| **风力** | 微风 |\n| **出行建议** | ✅ 适宜出行 
|\n\n**温馨提示：**\n- 当天天气晴朗，非常适合户外活动或出行安排\n- 
气温较为凉爽，早晚温差约6℃，建议随身携带一件薄外套 🧥\n- 
风力较小，整体体感舒适\n\n请问您还有其他日期或城市需要查询吗？😊',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 163,
                    'prompt_tokens': 512,
                    'total_tokens': 675,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 12,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 163
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 512
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-e77bb92